# MLflow and ML Pipeline Lab
Dataset: `student_placement.csv` (synthetic practice data). Run cells from top to bottom.

In [ ]:
%pip install -q pandas scikit-learn mlflow joblib

In [ ]:
import pandas as pd
df = pd.read_csv('student_placement.csv')
display(df.head())
print(df.shape)
print(df.isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mlflow, mlflow.sklearn
features = ['cgpa','coding_score','aptitude_score','projects','internships']
X, y = df[features], df['placed']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000))])

In [ ]:
mlflow.set_experiment('student-placement-notebook')
with mlflow.start_run():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = {'accuracy': accuracy_score(y_test,pred), 'precision': precision_score(y_test,pred,zero_division=0), 'recall': recall_score(y_test,pred,zero_division=0), 'f1': f1_score(y_test,pred,zero_division=0)}
    mlflow.log_param('model','LogisticRegression')
    mlflow.log_metrics({k: float(v) for k,v in metrics.items()})
    mlflow.sklearn.log_model(model, artifact_path='model')
print(metrics)

## Next experiments
Use `EXPERIMENTS.md` and `README.md` for the reusable pipeline, Prefect orchestration, optional Airflow DAG, and GitHub Actions CI workflow.